# Lab 04 — Delta Table Maintenance and Data Layout

Reliable Silver tables need more than correct rows. Their physical files must remain efficient as repeated `MERGE`, append, and schema-change operations create new files over time.

This notebook demonstrates a safe maintenance policy on an isolated copy of the Silver table:

- create a Delta table with **Liquid Clustering**;
- inspect its physical layout and Delta metadata;
- run `OPTIMIZE` to compact and incrementally cluster files;
- preview `VACUUM` with `DRY RUN`;
- run `VACUUM` with the safe seven-day retention period;
- prove that maintenance preserves row counts, unique keys, schema, and business values;
- compare Liquid Clustering with ZORDER and classic partitioning.

The production Silver table is read but never altered by this notebook. The maintenance target is a separate demonstration table, so reruns are deterministic and safe.


## 1. Load shared configuration

The configuration notebook provides the catalog, schema, production Silver table name, and validation flag. Run notebooks `00` through `10` before this notebook.


In [0]:
%run ./lab04_00_config


# Lab 04 — Configuration and Unity Catalog Setup

This notebook:
- defines Lab 4 parameters;
- creates the Unity Catalog catalog and schema when permitted;
- creates a managed or external volume;
- builds source, staging, landing, schema, checkpoint, quarantine, and test folders;
- defines all Bronze, Silver, SCD, and demonstration table names.

Run this notebook first. It is a setup notebook and does not need to be scheduled in the production Job.

Catalog ready: dbr_dev
Schema ready: dbr_dev.parvinbadalov
Volume ready: dbr_dev.parvinbadalov.lab04_silver_quality (external)
External location: abfss://parvinbadalov@dlspl21databricks.dfs.core.windows.net/lab04_silver_quality


Created or verified 13 Lab 4 folders under /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality
  source: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/source
  staging_initial: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial
  staging_incremental: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/incremental
  staging_evolved: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/evolved
  staging_invalid: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/invalid
  landing: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/landing
  schema_bronze: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/system/schema/bronze
  checkpoint_bronze: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/system/checkpoints/bronze
  checkpoint_silver: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/system/checkpoints/silver
  quarantine: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/quarantine
  schema_mismatch: /Volumes/dbr_dev/parvinba

Lab 4 table names configured:
  bronze: dbr_dev.parvinbadalov.lab04_bronze_retail
  silver_transactions: dbr_dev.parvinbadalov.lab04_silver_transactions
  quarantine: dbr_dev.parvinbadalov.lab04_quarantine
  quality_metrics: dbr_dev.parvinbadalov.lab04_quality_metrics
  product_scd0: dbr_dev.parvinbadalov.lab04_product_scd0
  product_scd1: dbr_dev.parvinbadalov.lab04_product_scd1
  product_scd2: dbr_dev.parvinbadalov.lab04_product_scd2
  product_scd3: dbr_dev.parvinbadalov.lab04_product_scd3
  product_scd4_current: dbr_dev.parvinbadalov.lab04_product_current
  product_scd4_history: dbr_dev.parvinbadalov.lab04_product_history
  product_scd6: dbr_dev.parvinbadalov.lab04_product_scd6
  schema_demo: dbr_dev.parvinbadalov.lab04_schema_demo
  column_mapping_demo: dbr_dev.parvinbadalov.lab04_column_mapping_demo


Upload the workbook to: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/source/Online Retail.xlsx
Expected columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']
Active contract: v1
Schema policy: fail
Trigger: availableNow; maximum files per trigger: 50


Volume validation succeeded; 6 top-level entries found.
dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/landing/
dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/quarantine/
dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/source/
dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/
dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/system/
dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/test_data/


In [0]:
from pyspark.sql import functions as F

silver_table = table_names["silver_transactions"]
maintenance_table = f"{catalog}.{schema}.lab04_maintenance_demo"
cluster_columns = ["invoice_date", "country"]
safe_vacuum_hours = 168

print(f"Production Silver source: {silver_table}")
print(f"Maintenance demo target: {maintenance_table}")
print(f"Liquid clustering columns: {cluster_columns}")
print(f"VACUUM retention: {safe_vacuum_hours} hours (7 days)")


Production Silver source: dbr_dev.parvinbadalov.lab04_silver_transactions
Maintenance demo target: dbr_dev.parvinbadalov.lab04_maintenance_demo
Liquid clustering columns: ['invoice_date', 'country']
VACUUM retention: 168 hours (7 days)


## 2. Verify the production Silver prerequisite

Maintenance must never conceal a data-quality problem. Before copying any data, this guard confirms that the Silver source exists and contains the stable transaction key and the columns selected for common query filters.


In [0]:
if not spark.catalog.tableExists(silver_table):
    raise FileNotFoundError(
        f"Silver table {silver_table} does not exist. "
        "Run lab04_04_silver_merge.ipynb first."
    )

source_df = spark.table(silver_table)
required_columns = {
    "transaction_line_id", "invoice_timestamp", "country",
    "invoice_no", "stock_code", "quantity", "unit_price",
}
missing_columns = sorted(required_columns - set(source_df.columns))
if missing_columns:
    raise AssertionError(f"Silver source is missing required columns: {missing_columns}")

source_metrics = source_df.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("transaction_line_id").alias("distinct_keys"),
    F.sum(F.col("transaction_line_id").isNull().cast("long")).alias("null_keys"),
).first().asDict()

if source_metrics["row_count"] == 0:
    raise AssertionError("Silver source is empty.")
if source_metrics["row_count"] != source_metrics["distinct_keys"]:
    raise AssertionError("Silver source contains duplicate transaction keys.")
if source_metrics["null_keys"] != 0:
    raise AssertionError("Silver source contains null transaction keys.")

print("✅ Silver prerequisite passed")
print(source_metrics)


✅ Silver prerequisite passed
{'row_count': 315101, 'distinct_keys': 315101, 'null_keys': 0}


## 3. Select an appropriate physical layout

Databricks recommends Liquid Clustering for new tables. This lab uses:

- `invoice_date` because time-range filters are common;
- `country` because geographic filtering is common and the column has moderate cardinality.

The table is deliberately **not partitioned** and does not use ZORDER. Liquid Clustering cannot be combined with either classic partitioning or ZORDER on the same table.

References: [Liquid Clustering](https://docs.databricks.com/aws/en/tables/clustering) and [table partitioning](https://docs.databricks.com/aws/en/tables/partitions).


In [0]:
silver_columns = spark.table(silver_table).columns

# The Silver table created earlier may already contain invoice_date.
# Reuse that column when present; otherwise derive it once from invoice_timestamp.
# This prevents two columns named invoice_date in the CTAS query.
if "invoice_date" in silver_columns:
    select_list = "*"
elif "invoice_timestamp" in silver_columns:
    select_list = "*, CAST(invoice_timestamp AS DATE) AS invoice_date"
else:
    raise AssertionError(
        "Silver must contain invoice_date or invoice_timestamp for maintenance clustering."
    )

spark.sql(f"DROP TABLE IF EXISTS {maintenance_table}")

spark.sql(f''' 
CREATE TABLE {maintenance_table}
USING DELTA
CLUSTER BY (invoice_date, country)
AS
SELECT
    {select_list}
FROM {silver_table}
''')

created_columns = spark.table(maintenance_table).columns
if created_columns.count("invoice_date") != 1:
    raise AssertionError("Maintenance table must contain exactly one invoice_date column.")

print(f"✅ Created clustered maintenance table: {maintenance_table}")


✅ Created clustered maintenance table: dbr_dev.parvinbadalov.lab04_maintenance_demo


## 4. Capture invariants and the pre-OPTIMIZE layout

`DESCRIBE DETAIL` exposes physical metrics such as file count and total bytes. Those values can vary by runtime and data size, so the notebook does not require a particular reduction. The hard requirements are business invariants: maintenance must preserve every row and key.


In [0]:
def table_detail(table_name):
    return spark.sql(f"DESCRIBE DETAIL {table_name}").first().asDict(recursive=True)


before_df = spark.table(maintenance_table)
before_metrics = before_df.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("transaction_line_id").alias("distinct_keys"),
    F.sum(
        F.xxhash64(*[F.col(c) for c in sorted(before_df.columns)])
        .cast("decimal(38,0)")
    )
        .alias("aggregate_record_hash"),
).first().asDict()

detail_before = table_detail(maintenance_table)
layout_before = {
    "phase": "before_optimize",
    "num_files": int(detail_before.get("numFiles") or 0),
    "size_bytes": int(detail_before.get("sizeInBytes") or 0),
}

print("Business invariants before maintenance:", before_metrics)
print("Physical layout before maintenance:", layout_before)


Business invariants before maintenance: {'row_count': 315101, 'distinct_keys': 315101, 'aggregate_record_hash': Decimal('-1629030752290624114929')}
Physical layout before maintenance: {'phase': 'before_optimize', 'num_files': 1, 'size_bytes': 23620052}


## 5. Run OPTIMIZE

`OPTIMIZE` rewrites Delta data files. On a Liquid Clustered table it both compacts files and organizes records by the clustering keys. The operation is incremental: after data changes, later runs process only files that need further organization.

Small lab tables may already contain one efficient file, so a second run can legitimately be a no-op. That is expected—not a failure.


In [0]:
optimize_result_df = spark.sql(f"OPTIMIZE {maintenance_table}")
display(optimize_result_df)
print("✅ OPTIMIZE completed")


path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 0, false, 0, 0, 1786318587879, 1786318593044, 8, 0, null, List(0, 0), null, 26, 26, 0, 0, List(23620052, true, false, false, null, null, null, null, 0, 0, 0, 0, 1, 23620052, 23620052, null, log, 16777216, 67108864, 4, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(218, 145, 0, 0, 0, 1763), 2, 1, 5, sizeAware, defaultSplitStrategy, null, false, 0, null, false, 0, 0, 0, null, null, null), null)"
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1786318593101, 1786318595364, 8, 0, null, List(0, 0), null, 26, 26, 0, 0, List(23620052, false, false, false, null, null, null, post-optimize-compaction, 0, 0, 0, 0, 0, 0, 0, null, null, 33554432, 67108864, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(0, 0, 1037, 0, 0, 0), 15, 1, 1, null, null, null, false, 0, null, false, 0, 0, 0, null, null, null), null)"


✅ OPTIMIZE completed


## 6. Validate the post-OPTIMIZE layout

The physical file count may decrease, remain unchanged, or occasionally change in another valid way as clustering rewrites files. Business data, however, must remain identical.


In [0]:
after_optimize_df = spark.table(maintenance_table)
after_optimize_metrics = after_optimize_df.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("transaction_line_id").alias("distinct_keys"),
    F.sum(
        F.xxhash64(*[F.col(c) for c in sorted(after_optimize_df.columns)])
        .cast("decimal(38,0)")
    )
        .alias("aggregate_record_hash"),
).first().asDict()

detail_after_optimize = table_detail(maintenance_table)
layout_after_optimize = {
    "phase": "after_optimize",
    "num_files": int(detail_after_optimize.get("numFiles") or 0),
    "size_bytes": int(detail_after_optimize.get("sizeInBytes") or 0),
}

if after_optimize_metrics != before_metrics:
    raise AssertionError(
        f"OPTIMIZE changed business invariants: before={before_metrics}, "
        f"after={after_optimize_metrics}"
    )

layout_comparison_df = spark.createDataFrame(
    [layout_before, layout_after_optimize]
)
display(layout_comparison_df.orderBy("phase"))
print("✅ OPTIMIZE preserved rows, keys, and aggregate record hash")


num_files,phase,size_bytes
1,after_optimize,23620052
1,before_optimize,23620052


✅ OPTIMIZE preserved rows, keys, and aggregate record hash


## 7. Preview VACUUM safely

`VACUUM ... DRY RUN` lists files eligible for deletion without deleting them. This is the review step that should precede destructive maintenance in a production workflow.

The default and recommended minimum retention is seven days. Shorter retention can remove files still needed by concurrent jobs and permanently reduce the available time-travel window.

Reference: [VACUUM](https://docs.databricks.com/aws/en/tables/operations/vacuum).


In [0]:
vacuum_preview_df = spark.sql(
    f"VACUUM {maintenance_table} RETAIN {safe_vacuum_hours} HOURS DRY RUN"
)
preview_count = vacuum_preview_df.count()

print(f"VACUUM DRY RUN found {preview_count:,} eligible files")
display(vacuum_preview_df)


VACUUM DRY RUN found 0 eligible files


path


## 8. Run VACUUM with safe retention

This command uses 168 hours and keeps the Delta retention safety check enabled. It never disables the safety guard and never uses zero-hour retention.

`VACUUM` removes unreferenced physical files; it does not delete current table rows. After it completes, versions older than the retention window may no longer be readable because their data files can be gone.


In [0]:
vacuum_result_df = spark.sql(
    f"VACUUM {maintenance_table} RETAIN {safe_vacuum_hours} HOURS"
)
display(vacuum_result_df)
print("✅ Safe VACUUM completed")


path
""


✅ Safe VACUUM completed


## 9. Validate after VACUUM and prove rerun safety

The table is queried again after cleanup. The notebook then runs `OPTIMIZE` a second time to demonstrate that a scheduled rerun remains safe when no new files require work.


In [0]:
after_vacuum_df = spark.table(maintenance_table)
after_vacuum_metrics = after_vacuum_df.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("transaction_line_id").alias("distinct_keys"),
    F.sum(
        F.xxhash64(*[F.col(c) for c in sorted(after_vacuum_df.columns)])
        .cast("decimal(38,0)")
    )
        .alias("aggregate_record_hash"),
).first().asDict()

if after_vacuum_metrics != before_metrics:
    raise AssertionError(
        f"VACUUM changed business invariants: before={before_metrics}, "
        f"after={after_vacuum_metrics}"
    )

second_optimize_df = spark.sql(f"OPTIMIZE {maintenance_table}")
display(second_optimize_df)

final_metrics = spark.table(maintenance_table).agg(
    F.count("*").alias("row_count"),
    F.countDistinct("transaction_line_id").alias("distinct_keys"),
    F.sum(
        F.xxhash64(*[F.col(c) for c in sorted(after_vacuum_df.columns)])
        .cast("decimal(38,0)")
    )
        .alias("aggregate_record_hash"),
).first().asDict()

if final_metrics != before_metrics:
    raise AssertionError("The second OPTIMIZE run changed business data.")

print("✅ VACUUM and repeated OPTIMIZE preserved all business invariants")
print(final_metrics)


path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 0, false, 0, 0, 1786318621044, 1786318623393, 8, 0, null, List(0, 0), null, 26, 26, 0, 0, List(23620052, false, false, false, null, null, null, null, 0, 0, 0, 0, 1, 23620052, 23620052, null, log, 16777216, 67108864, 4, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(76, 6, 682, 0, 0, 0), 2, 1, 5, sizeAware, null, null, false, 0, null, false, 0, 0, 0, null, null, null), null)"
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1786318623423, 1786318624449, 8, 0, null, List(0, 0), null, 26, 26, 0, 0, List(23620052, false, false, false, null, null, null, post-optimize-compaction, 0, 0, 0, 0, 0, 0, 0, null, null, 33554432, 67108864, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(0, 0, 410, 0, 0, 0), 15, 1, 1, null, null, null, false, 0, null, false, 0, 0, 0, null, null, null), null)"


✅ VACUUM and repeated OPTIMIZE preserved all business invariants
{'row_count': 315101, 'distinct_keys': 315101, 'aggregate_record_hash': Decimal('-1629030752290624114929')}


## 10. Confirm Liquid Clustering metadata

`DESCRIBE DETAIL` should report the selected clustering columns. The table must remain unpartitioned because partitioning and Liquid Clustering are mutually exclusive.


In [0]:
final_detail = table_detail(maintenance_table)
final_clustering_columns = final_detail.get("clusteringColumns") or []
final_partition_columns = final_detail.get("partitionColumns") or []

if set(final_clustering_columns) != set(cluster_columns):
    raise AssertionError(
        f"Unexpected clustering columns: {final_clustering_columns}; "
        f"expected {cluster_columns}"
    )
if final_partition_columns:
    raise AssertionError(
        f"Liquid Clustered table must not be partitioned: {final_partition_columns}"
    )

metadata_summary_df = spark.createDataFrame([{
    "table_name": maintenance_table,
    "format": final_detail.get("format"),
    "num_files": int(final_detail.get("numFiles") or 0),
    "size_bytes": int(final_detail.get("sizeInBytes") or 0),
    "clustering_columns": ", ".join(final_clustering_columns),
    "partition_columns": ", ".join(final_partition_columns) or "none",
    "vacuum_retention_hours": safe_vacuum_hours,
}])

display(metadata_summary_df)
print("✅ Liquid Clustering metadata validated")


clustering_columns,format,num_files,partition_columns,size_bytes,table_name,vacuum_retention_hours
"invoice_date, country",delta,1,none,23620052,dbr_dev.parvinbadalov.lab04_maintenance_demo,168


✅ Liquid Clustering metadata validated


## 11. Compare physical-layout strategies

| Strategy | Best fit | Advantages | Limitations | Lab decision |
|---|---|---|---|---|
| Liquid Clustering | New or evolving Delta tables with changing filter patterns | Flexible keys, incremental clustering, avoids rigid partition boundaries | Requires `OPTIMIZE`; cannot combine with partitioning or ZORDER | **Selected** |
| ZORDER | Existing non-clustered Delta tables with stable high-cardinality filter columns | Improves data skipping without partitioning | Legacy choice for new tables; effectiveness falls with too many columns; cannot combine with Liquid Clustering | Discussed only |
| Classic partitioning | Very large tables with stable, low-cardinality predicates and enough data in every partition | Strong partition pruning | Small files and skew from poor keys; boundaries are rigid; high-cardinality keys are harmful | Not selected |
| No explicit layout | Small tables or automatically managed workloads | Simplest operations | Less control for large or highly selective workloads | Acceptable for small tables |

`OPTIMIZE` is a maintenance operation, not a competing layout strategy. `VACUUM` controls storage cleanup and time-travel retention; it does not improve clustering by itself.


## 12. Final validation and Delta history

This final evidence block summarizes functional correctness and maintenance configuration. Capture the summary and history for the README.


In [0]:
validation_results = [
    ("source_and_target_rows_match", int(final_metrics["row_count"] == source_metrics["row_count"])),
    ("unique_transaction_keys", int(final_metrics["row_count"] == final_metrics["distinct_keys"])),
    ("optimize_preserved_record_hash", int(final_metrics["aggregate_record_hash"] == before_metrics["aggregate_record_hash"])),
    ("liquid_clustering_enabled", int(set(final_clustering_columns) == set(cluster_columns))),
    ("classic_partitions_absent", int(len(final_partition_columns) == 0)),
    ("safe_vacuum_retention", int(safe_vacuum_hours >= 168)),
]

validation_df = spark.createDataFrame(
    validation_results,
    ["validation_rule", "passed"],
)
failed_rules = [row["validation_rule"] for row in validation_df.filter("passed = 0").collect()]
if failed_rules:
    raise AssertionError(f"Maintenance validation failed: {failed_rules}")

display(validation_df.orderBy("validation_rule"))
display(spark.sql(f"DESCRIBE HISTORY {maintenance_table}"))
print("✅ Lab 04 table-maintenance validation passed")


validation_rule,passed
classic_partitions_absent,1
liquid_clustering_enabled,1
optimize_preserved_record_hash,1
safe_vacuum_retention,1
source_and_target_rows_match,1
unique_transaction_keys,1


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-08-09T23:37:00.000Z,75952854126293,parvinbadalov@yahoo.com,VACUUM END,Map(status -> COMPLETED),null,List(1151279916123287),e90e1914-cdef-419b-9677-e8e11f70024c,0809-191624-8qt42caq-v2n,2,SnapshotIsolation,true,"Map(numDeletedFiles -> 0, numVacuumedDirectories -> 1)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-08-09T23:36:59.000Z,75952854126293,parvinbadalov@yahoo.com,VACUUM START,"Map(retentionCheckEnabled -> true, defaultRetentionMillis -> 604800000)",null,List(1151279916123287),e90e1914-cdef-419b-9677-e8e11f70024c,0809-191624-8qt42caq-v2n,1,SnapshotIsolation,true,"Map(numFilesToDelete -> 0, sizeOfDataToDelete -> 0)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-08-09T23:36:32.000Z,75952854126293,parvinbadalov@yahoo.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [""invoice_date"",""country""], isFull -> false, zOrderBy -> [], batchId -> -1)",null,List(1151279916123287),74f1811f-9acf-482f-8460-7ff44630891a,0809-191624-8qt42caq-v2n,0,SnapshotIsolation,true,Map(),null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-08-09T23:36:25.000Z,75952854126293,parvinbadalov@yahoo.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [""invoice_date"",""country""], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""databricks.delta.expressionStats.selectedColumns"":""upper(country),lower(country)"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-d877537d-6e85-4853-b2db-784379c7a901"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-9b750cb0-a868-4779-a79d-066befaa3083""}, statsOnLoad -> false, clusteringOnWriteStatus -> Reason for skipping: Estimated ingestion size is not within the expected range)",null,List(1151279916123287),10a46b6c-a9cf-4678-b113-bf4d9c0e66a8,0809-191624-8qt42caq-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 315101, numOutputBytes -> 23620052)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


✅ Lab 04 table-maintenance validation passed


## What this notebook proved

- Liquid Clustering was configured on `invoice_date` and `country`.
- The maintenance table has no classic partitions and no ZORDER configuration.
- `OPTIMIZE` completed and preserved all rows, unique keys, and aggregate business values.
- `VACUUM DRY RUN` provided a non-destructive preview.
- `VACUUM` ran with the safe 168-hour retention period and preserved current data.
- A repeated `OPTIMIZE` run remained safe, supporting scheduled execution.
- The trade-offs among Liquid Clustering, ZORDER, and partitioning were documented.

### Suggested README evidence

Capture:

1. the before/after physical-layout table;
2. the `VACUUM DRY RUN` result;
3. the Liquid Clustering metadata summary;
4. the final validation table;
5. the Delta history output.

## Next notebook

Run **`lab04_12_validation.ipynb`** next. It will validate the complete Lab 4 solution across Silver quality, idempotent MERGE, SCD tables, controlled schema evolution, column mapping, and maintenance configuration.
